# Extraksi Fitur Wajah

In [1]:
import utils as u
import torch
from transformers import AutoModelForImageClassification
from tqdm import tqdm
import pandas as pd
import joblib
import numpy as np
import time
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## Load Model

In [2]:
model_path = 'skutaada/VIT-VGGFace'
model = AutoModelForImageClassification.from_pretrained(model_path).to(device)
model_path

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

'skutaada/VIT-VGGFace'

## DemogPairs

### Load Dataset

In [3]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx,subject_id
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5,able_wanamakok
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5,able_wanamakok
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5,able_wanamakok
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5,able_wanamakok
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5,able_wanamakok
...,...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3,zachary_quinto
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3,zachary_quinto
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3,zachary_quinto
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3,zachary_quinto


### Ektraksi Fitur

In [4]:
os.makedirs('features', exist_ok=True)
os.makedirs('times', exist_ok=True)

features = {}
times = {}

start_total = time.perf_counter()
for d in tqdm(data):
    t0 = time.perf_counter()
    features[d['image_path']] = u.extract_vit_features(d['full_path'], model=model, model_path=model_path, device=device)
    t1 = time.perf_counter()
    times[d['image_path']] = t1 - t0

total_time = time.perf_counter() - start_total
avg_time = total_time / len(data) if len(data) > 0 else 0

timing_data = {
    'model_name': model_path,
    'feature_name': 'vit-face',
    'device': str(device),
    'total_images': len(data),
    'total_time_seconds': round(total_time, 4),
    'total_time_formatted': u.seconds_to_time(total_time)['text'],
    'avg_time_per_image_seconds': round(avg_time, 6),
    'avg_time_per_image_ms': round(avg_time * 1000, 4),
    'times_per_image': times
}

u.save_json(timing_data, 'times/features_vit-face.json')
features[d['image_path']].shape, joblib.dump(features, 'features/demogpairs_vit-face.pkl', compress=9)

  0%|                                                                                                                                                                                                      | 0/10800 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10800/10800 [07:09<00:00, 25.16it/s]


((768,), ['features/demogpairs_vit-face.pkl'])